# Scoreboard

**Goal:** one honest, repeatable way to measure the whole system, so every improvement in round 2 is judged
**before vs after** on the same nights with the same definitions.

Read `docs/scoreboard.md` alongside this notebook.

In [ ]:
import subprocess
from datetime import date

import numpy as np
import pandas as pd
import torch

from asthunt.config.load_config import load_config
from asthunt.dataset.make_unit_tag import make_unit_tag
from asthunt.acquire.load_unit_metadata import load_unit_metadata
from asthunt.acquire.query_ztf_metadata import query_ztf_metadata
from asthunt.models.load_model_bundle import load_model_bundle
from asthunt.pipeline.run_night_pipeline import run_night_pipeline
from asthunt.evaluate.build_scoreboard import build_scoreboard
from asthunt.evaluate.save_scoreboard_rows import save_scoreboard_rows

project = load_config("project"); paths = project["paths"]
configs = {name: load_config(name) for name in ["acquire", "inject", "detect", "link"]}
acquire = configs["acquire"]


def unit_of(night):
    """The original quadrant (configs/acquire.yaml: field, ccd, quadrant) on one night."""
    return {"field": acquire["field"], "ccd": acquire["ccd"], "quadrant": acquire["quadrant"], "night": night}


metadata = load_unit_metadata(unit_of(None), paths["raw"])
device = "mps" if torch.backends.mps.is_available() else "cpu"
# Every saved bundle (models/v1, models/v2, ...) is evaluated the same way
VERSIONS = sorted(folder.name for folder in paths["models"].iterdir() if folder.is_dir() and (folder / "bundle.json").exists())
for version in VERSIONS:
    print(version, ":", load_model_bundle(paths["models"], version)["info"]["description"])

## 1. The evaluation nights
Each split now covers every **unit** (CCD quadrant × night): 3 quadrants of field 618 per night.
- **validation**: the 2 nights used to choose thresholds (expect these numbers to be a little optimistic)
- **test**: 2019-02-08 (opened while debugging in round 1; still never trained on)
- **holdout**: 2022-06-17, opened **only here**, only to produce numbers. Never used to choose anything.

In [ ]:
from asthunt.acquire.list_units import list_units
from asthunt.dataset.make_unit_rng import make_unit_rng

# Every unit (CCD quadrant x night) of each split: the original quadrant, the extra quadrants, extra fields
splits = {split: list_units(acquire, split) for split in ["validation", "test", "holdout"]}
for split, units in splits.items():
    print(f"{split:10s}: {len(units)} units")


def evaluate_unit(unit, bundle):
    # The SAME pipeline as the real search, with fakes planted, and no report written
    return run_night_pipeline(unit, load_unit_metadata(unit, paths["raw"]), configs, bundle, paths,
                              make_unit_rng(project["seed"], unit), device, with_injections=True, write_reports=False)


results = {}
for version in VERSIONS:
    bundle = load_model_bundle(paths["models"], version)
    results[version] = {split: [evaluate_unit(u, bundle) for u in units] for split, units in splits.items()}
    print(version, "evaluated")

## 2. One row per split

In [ ]:
git_sha = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=paths["models"].parent).stdout.strip()
rows = []
for version in VERSIONS:
    thresholds = load_model_bundle(paths["models"], version)["thresholds"]
    for split, night_results in results[version].items():
        # min_detections here defines which KNOWN asteroids count as findable (present in >= 4 frames):
        # a fixed definition, the same for every version
        row = build_scoreboard(night_results, thresholds, configs["link"]["min_detections"],
                               configs["inject"]["known_asteroid_max_v_mag"])
        rows.append({"version": version, "split": split, "evaluated": str(date.today()), "code": git_sha, **row})

scoreboard = save_scoreboard_rows(rows, paths["reports"] / "scoreboard.csv")
scoreboard.round(3)

## 3. How to read it
| column | good direction | meaning |
|---|---|---|
| `nights` | | number of units (quadrant-nights) evaluated |
| `detection_ap` | ↑ | how well the CNN ranks real above bogus detections |
| `detection_recall` | ↑ | real detections surviving the CNN cut (lost here = lost forever) |
| `bogus_kept_per_frame` | ↓ | junk the linker has to sift through |
| `tracklet_ap` | ↑ | how well tracklets are ranked (fakes and known asteroids count as real) |
| `mag_at_50_completeness` | ↑ (fainter) | **depth**: planted asteroids found half the time at this magnitude |
| `known_recovered` | ↑ | real known asteroids (V ≤ 20.5) found / present |
| `unexplained_per_night` | ↓ (but not 0 by force) | accepted tracklets nobody can explain, per unit (one quadrant-night): a human's vetting load |

In [ ]:
for version in VERSIONS:
    holdout = scoreboard[(scoreboard["version"] == version) & (scoreboard["split"] == "holdout")].iloc[0]
    print(f"{version} on the holdout night: depth {holdout['mag_at_50_completeness']:.2f} mag, "
          f"known asteroids {holdout['known_recovered']}, unexplained alarms per night {holdout['unexplained_per_night']:.1f}")

## Exercises
1. Which split has the deepest `mag_at_50_completeness`? Use the nights' seeing and depth (`maglimit`) to explain why.
2. Why is `unexplained_per_night` not automatically "bad"? (Hint: what are we hunting for?)
3. Add a column of your own to `build_scoreboard` (e.g. completeness at mag 21). Re-run: v1's row updates, and nothing is duplicated.